# Lab 13 — Part 2: Search Functions and Retrieval Approaches

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [2]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for i in top_ids:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

search("How can we measure similarity between text vectors?")

Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [3]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
--- Relevant query ---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


**Observation:** both queries return two documents, but for *"volcano eruption"* every score is **0.000**, because none of its words appear in the collection. The two documents shown are only there because the function always returns the top *k*, in an arbitrary order. For *"cosine similarity"* the best document has a clear positive score (about 0.594), and the second one is again 0. A high rank therefore does not guarantee a good match, so the score has to be checked as well.

## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [4]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.


**Observation:** `search_filtered` prints *"No matching documents found."* for *"volcano eruption"* and shows only the one real match for *"cosine similarity"*, so the zero-score documents no longer appear as fake results.

## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

In [5]:
# Practice: run your own queries here
search("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


**Answer (Practice 1):** the document *"Named entity recognition identifies names, places, and organizations in text."* ranks first, with a score of about 0.625. The second result has a score of 0.000, so it is not relevant: it appears only because `k=2`.

In [6]:
# Practice 2 - add a new document
documents.append("BERT is a transformer model that uses bidirectional context.")

# Before rebuilding: the vectorizer and the document matrix still describe the OLD 6 documents
print("Before rebuilding (matrix shape:", document_vectors.shape, ")")
search("transformer models", k=3)

# Rebuild the TF-IDF vectors for the new, 7-document collection
document_vectors = vectorizer.fit_transform(documents)
print("\nAfter rebuilding (matrix shape:", document_vectors.shape, ")")
search("transformer models", k=3)

Before rebuilding (matrix shape: (6, 39) )
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
Score: 0.000 | Named entity recognition identifies names, places, and organizations in text.

After rebuilding (matrix shape: (7, 45) )
Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


**Answer (Practice 2):** rebuilding was necessary because the vectorizer learns its **vocabulary and IDF weights** when it is fitted, and `document_vectors` is a fixed matrix with one row for each of the **old** documents. Adding a string to the `documents` list changes neither of them: the new document has no vector, so it can never be returned, and its words (*bert*, *model*, *bidirectional*) are not in the vocabulary. Calling `vectorizer.fit_transform(documents)` again learns the new vocabulary, recomputes the IDF weights (they depend on how many documents contain each word) and creates a vector for every document.

The results show this: before rebuilding, every score for *"transformer models"* is 0.000, and after rebuilding the new BERT document ranks first (about 0.408). Note that even the original *"Transformers use self-attention..."* note does not match the query, because TF-IDF compares exact words and *transformer* is not the same word as *transformers*. Only the new document contains the singular form.